# One Piece Chapter 1177 — reMarkable 2 Landscape Spread

This notebook downloads One Piece Ch.1177 from WeebCentral and creates a **landscape double-page spread PDF** optimized for reMarkable 2.

- **Smart spread detection**: wide source images (already-combined double spreads) get their own full sheet
- Portrait pages paired naturally: [1,2], [3,4], [5,6]... — spreads align automatically
- Resolution: 1872×1404 @ 226 DPI (native reMarkable 2)

**Just run all cells!** No input needed.

In [ ]:
# Cell 1: Install dependencies & clone repo
!pip install -q requests 'httpx[http2]' nest_asyncio beautifulsoup4 lxml Pillow fpdf2 tqdm
!rm -rf /content/weebcentral_downloader
!git clone https://github.com/Yui007/weebcentral_downloader
print('\n✅ Setup complete!')

In [ ]:
# Cell 2: Download Chapter 1177
import sys
sys.path.insert(0, '/content/weebcentral_downloader/colab')

from colab_scraper import scrape_manga_info, scrape_chapter_list
from colab_downloader import download_chapters, parse_chapter_selection

SERIES_URL = 'https://weebcentral.com/series/01J76XY7E9FNDZ1DBBM6PBJPFK/One-Piece'
CHAPTER_NUM = 1177

manga_info = scrape_manga_info(SERIES_URL)
chapters = scrape_chapter_list(SERIES_URL)

# Find chapter 1177 by index
selected = [CHAPTER_NUM - 1]  # 0-based index
print(f'\n📖 Selected: {chapters[selected[0]]["title"]}')

# Download as images (we'll make our own PDF)
output_dir = download_chapters(
    manga_info=manga_info,
    chapters=chapters,
    selected_indices=selected,
    output_format='images',
    output_dir='/content/manga',
)
print(f'\n✅ Chapter downloaded to: {output_dir}')

In [ ]:
# Cell 3: Create landscape double-page spread PDF for reMarkable 2
import os
import glob
from PIL import Image

# reMarkable 2 native resolution in landscape
LANDSCAPE_WIDTH = 1872
LANDSCAPE_HEIGHT = 1404


def is_wide(filepath):
    """Check if an image is landscape/wide (an already-combined double spread)."""
    with Image.open(filepath) as img:
        return img.width > img.height


def scale_to_fit(img, max_width, max_height):
    """Scale image to fit within max dimensions, preserving aspect ratio."""
    ratio = min(max_width / img.width, max_height / img.height)
    if ratio >= 1:
        return img
    new_size = (int(img.width * ratio), int(img.height * ratio))
    return img.resize(new_size, Image.LANCZOS)


def create_landscape_spread(image_dir, output_path):
    """
    Create landscape PDF with smart spread detection.
    - Wide source images (already-combined double spreads) get their own FULL sheet
    - Portrait pages paired naturally: [1,2], [3,4], [5,6]...
    - Manga reading order: right page first, left page second
    """
    image_files = sorted(glob.glob(os.path.join(image_dir, '*')))
    image_files = [f for f in image_files
                   if f.lower().endswith(('.png', '.jpg', '.jpeg', '.webp', '.gif'))]

    if not image_files:
        print('ERROR: No images found!')
        return

    print(f'Found {len(image_files)} pages. Detecting spreads...')

    # Classify each page
    wide_pages = set()
    for f in image_files:
        if is_wide(f):
            wide_pages.add(f)
            print(f'   Wide spread detected: {os.path.basename(f)}')

    # Build sheet plan:
    # - Wide images -> solo full sheet
    # - Portrait images -> pair [1,2], [3,4], etc.
    sheets_plan = []  # list of (right_path, left_path) or (wide_path, 'FULL')
    portrait_buffer = []

    def flush_portraits():
        nonlocal portrait_buffer
        if not portrait_buffer:
            return
        i = 0
        while i < len(portrait_buffer):
            right = portrait_buffer[i]
            left = portrait_buffer[i + 1] if i + 1 < len(portrait_buffer) else None
            sheets_plan.append((right, left))
            i += 2
        portrait_buffer = []

    for f in image_files:
        if f in wide_pages:
            flush_portraits()
            sheets_plan.append((f, 'FULL'))
        else:
            portrait_buffer.append(f)

    flush_portraits()

    half_w = LANDSCAPE_WIDTH // 2
    sheets = []

    for right_path, left_path in sheets_plan:
        sheet = Image.new('RGB', (LANDSCAPE_WIDTH, LANDSCAPE_HEIGHT), (255, 255, 255))

        if left_path == 'FULL':
            # Wide spread — scale to fill the entire sheet
            wide_img = Image.open(right_path).convert('RGB')
            wide_img = scale_to_fit(wide_img, LANDSCAPE_WIDTH, LANDSCAPE_HEIGHT)
            x = (LANDSCAPE_WIDTH - wide_img.width) // 2
            y = (LANDSCAPE_HEIGHT - wide_img.height) // 2
            sheet.paste(wide_img, (x, y))
            wide_img.close()
        else:
            # Right half (read first in manga)
            right_img = Image.open(right_path).convert('RGB')
            right_img = scale_to_fit(right_img, half_w, LANDSCAPE_HEIGHT)
            x = half_w + (half_w - right_img.width) // 2
            y = (LANDSCAPE_HEIGHT - right_img.height) // 2
            sheet.paste(right_img, (x, y))
            right_img.close()

            # Left half (read second in manga)
            if left_path:
                left_img = Image.open(left_path).convert('RGB')
                left_img = scale_to_fit(left_img, half_w, LANDSCAPE_HEIGHT)
                x = (half_w - left_img.width) // 2
                y = (LANDSCAPE_HEIGHT - left_img.height) // 2
                sheet.paste(left_img, (x, y))
                left_img.close()

        sheets.append(sheet)

    # Save as PDF
    sheets[0].save(
        output_path,
        'PDF',
        resolution=226.0,
        save_all=True,
        append_images=sheets[1:],
    )
    for s in sheets:
        s.close()

    wide_count = sum(1 for _, lp in sheets_plan if lp == 'FULL')
    paired_count = len(sheets_plan) - wide_count
    print(f'\n✅ Landscape spread PDF created: {output_path}')
    print(f'   {len(sheets)} sheets total')
    print(f'   {wide_count} full-width double spreads')
    print(f'   {paired_count} paired portrait sheets')


# Find the chapter image directory
manga_dir = output_dir
image_dirs = []
for entry in sorted(os.listdir(manga_dir)):
    entry_path = os.path.join(manga_dir, entry)
    if os.path.isdir(entry_path):
        images = [f for f in os.listdir(entry_path)
                  if f.lower().endswith(('.png', '.jpg', '.jpeg', '.webp'))]
        if images:
            image_dirs.append(entry_path)

if not image_dirs:
    print('ERROR: No image directories found!')
else:
    chapter_image_dir = image_dirs[-1]  # Latest/only chapter
    print(f'Using images from: {chapter_image_dir}')

    output_pdf = os.path.join(manga_dir, 'One_Piece_Ch1177_Landscape_Spread.pdf')
    create_landscape_spread(chapter_image_dir, output_pdf)

In [ ]:
# Cell 4: Download the PDF to your device
from google.colab import files

pdf_path = os.path.join(output_dir, 'One_Piece_Ch1177_Landscape_Spread.pdf')
if os.path.exists(pdf_path):
    print(f'📥 Downloading: {os.path.basename(pdf_path)}')
    print(f'   Size: {os.path.getsize(pdf_path) / 1024 / 1024:.1f} MB')
    files.download(pdf_path)
else:
    print('❌ PDF not found. Make sure Cell 3 ran successfully.')